In [1]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [2]:
from rag_helper import RAGBase
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [3]:
instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
    instructions=instructions,
)

In [4]:
answer = assistant.rag("How do I run Ollama locally?")
print(answer)

To run Ollama locally, install it first from [https://ollama.com/download](https://ollama.com/download) for your operating system:

- **macOS**: download the `.pkg` and install it
- **Windows**: download the `.msi` and install it
- **Linux**: run:
  ```bash
  curl -fsSL https://ollama.com/install.sh | sh
  ```

Then open a terminal and run:

```bash
ollama run llama3
```

This will download the LLaMA 3 model, start it locally, and open a chat-like interface.

To check that the local server is running, you can also run:

```bash
curl http://localhost:11434
```

If needed, you can restart the Ollama server with:

```bash
!nohup ollama serve > nohup.out 2>&1 &
```


In [5]:
answer = assistant.rag("How do I run Olama locally?")
print(answer)

I don’t see any FAQ entry about running **Ollama/Olama** locally.

The closest related guidance is that you **can run the course locally instead of Codespaces** if you’re comfortable setting up the needed tools, and you should **document your setup and keep it reproducible**.

If you meant the **course environment** rather than Ollama specifically, that’s the available guidance in the FAQ.


In [6]:
messages = [
    {"role": "user", "content": "I just discovered the course. Can I join it?"}
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
)

answer = response.output_text
print(answer)

Yes, most likely — but it depends on the course’s enrollment policy and whether it’s still open.

If you want, I can help you figure it out quickly. Usually the next steps are:

1. Check the course page for:
   - enrollment deadline
   - prerequisites
   - whether late enrollment is allowed

2. Contact the instructor or course admin and ask:
   - “I just discovered this course. Is it still possible to join?”
   - “Are there any prerequisites or catch-up steps I should know about?”

If you’d like, I can also help you write a short message to ask to join.


In [7]:
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [8]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [9]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

response.output

[ResponseFunctionToolCall(arguments='{"query":"join course discovered can I join late enrollment FAQ"}', call_id='call_6QNyEBsKGMKAArItaGPNV6AS', name='search', type='function_call', id='fc_0e8d72c477357864006a36e2e1dc48819893131975ddd643ce', namespace=None, status='completed')]

In [10]:
import json

call = response.output[0]
args = json.loads(call.arguments)

results = search(**args)
result_json = json.dumps(results, indent=2)

In [11]:
messages.extend(response.output)

messages.append({
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": result_json,
})

In [12]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

answer = response.output_text
print(answer)

Yes — you can still join and start learning.

If you want a certificate, the important thing is to submit your project while the course is still accepting submissions.


In [13]:
usage = response.usage
usage.input_tokens, usage.output_tokens

(770, 36)

In [14]:
def calculate_gpt54mini_price(input_tokens, output_tokens):
    INPUT_PRICE_PER_MILLION = 0.15
    OUTPUT_PRICE_PER_MILLION = 0.60

    input_cost = (input_tokens / 1_000_000) * INPUT_PRICE_PER_MILLION
    output_cost = (output_tokens / 1_000_000) * OUTPUT_PRICE_PER_MILLION
    total_cost = input_cost + output_cost

    return {
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": total_cost,
    }

result = calculate_gpt54mini_price(652, 33)
print("Total cost: $", round(result["total_cost"], 8))

Total cost: $ 0.0001176


In [15]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

In [16]:
def make_call(call):
    args = json.loads(call.arguments)

    if call.name == "search":
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        "call_id": call.call_id,
        "output": result_json,
    }

In [17]:
question = "I just discovered the course. Can I join it?"

messages = [
    {"role": "developer", "content": instructions},
    {"role": "user", "content": question},
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

messages.extend(response.output)
has_function_calls = False

for item in response.output:
    if item.type == "function_call":
        print("function_call:", item.name, item.arguments)
        call_output = make_call(item)
        messages.append(call_output)
        has_function_calls = True

    elif item.type == "message":
        print("ASSISTANT:")
        print(item.content[0].text)

function_call: search {"query":"join course discovered course can I join enrollment registration late join discovered the course"}


In [19]:
def agent_loop(instructions, question, model="gpt-5.4-mini") -> str:
    messages = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": question}
    ]

    it = 1

    while True:
        print(f"iteration #{it}...")
        has_function_calls = False

        response = openai_client.responses.create(
            model=model,
            input=messages,
            tools=[search_tool]
        )

        messages.extend(response.output)

        for item in response.output:
            if item.type == "function_call":
                print("function_call:", item.name, item.arguments)
                call_output = make_call(item)
                messages.append(call_output)
                has_function_calls = True

            elif item.type == "message":
                print("ASSISTANT:")
                last_answer = item.content[0].text
                print(item.content[0].text)

        it = it + 1
        if has_function_calls == False:
            break

    return last_answer

In [20]:
agent_loop(instructions, "How do I run Olama locally?")

iteration #1...
function_call: search {"query":"Olama local run install run locally Ollama"}
function_call: search {"query":"Ollama run locally setup models local machine"}
function_call: search {"query":"how to run Ollama locally course FAQ"}
iteration #2...
ASSISTANT:
To run **Ollama locally**:

1. **Install Ollama**
   - Go to: https://ollama.com/download
   - Choose your OS:
     - **macOS**: download the `.pkg`
     - **Windows**: download the `.msi`
     - **Linux**: run:
       ```bash
       curl -fsSL https://ollama.com/install.sh | sh
       ```

2. **Start a model locally**
   ```bash
   ollama run llama3
   ```
   This will download the model and open a local chat interface.

3. **Check that the local server is running**
   ```bash
   curl http://localhost:11434
   ```
   You should get a response from Ollama.

4. **Use it from Python**
   Install the client:
   ```bash
   pip install ollama
   ```

   Example:
   ```python
   import ollama

   response = ollama.chat(
     

'To run **Ollama locally**:\n\n1. **Install Ollama**\n   - Go to: https://ollama.com/download\n   - Choose your OS:\n     - **macOS**: download the `.pkg`\n     - **Windows**: download the `.msi`\n     - **Linux**: run:\n       ```bash\n       curl -fsSL https://ollama.com/install.sh | sh\n       ```\n\n2. **Start a model locally**\n   ```bash\n   ollama run llama3\n   ```\n   This will download the model and open a local chat interface.\n\n3. **Check that the local server is running**\n   ```bash\n   curl http://localhost:11434\n   ```\n   You should get a response from Ollama.\n\n4. **Use it from Python**\n   Install the client:\n   ```bash\n   pip install ollama\n   ```\n\n   Example:\n   ```python\n   import ollama\n\n   response = ollama.chat(\n       model=\'llama3\',\n       messages=[{"role": "user", "content": "Hello!"}]\n   )\n\n   print(response[\'message\'][\'content\'])\n   ```\n\nIf you get a **connection refused** error, restart the server with:\n```bash\nollama serve\n`

In [21]:
agent_loop(instructions, "I just discovered the course. Can I still join it?")

iteration #1...
function_call: search {"query":"can I still join course discovered course join late enrollment"}
iteration #2...
ASSISTANT:
Yes, you can still join the course.

If you want a certificate, make sure you submit your project while submissions are still being accepted. If you’re just learning, you can also start at any time using the available videos and materials.

Would you like help with how to get started or how the weekly workflow works?


'Yes, you can still join the course.\n\nIf you want a certificate, make sure you submit your project while submissions are still being accepted. If you’re just learning, you can also start at any time using the available videos and materials.\n\nWould you like help with how to get started or how the weekly workflow works?'

In [28]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches.

The question has to be about the course or its logistics, offtopic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the 
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
""".strip()


In [29]:
def agent_loop(instructions, question, model="gpt-5.4-mini") -> str:
    messages = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": question}
    ]

    it = 1

    while True:
        print(f"iteration #{it}...")
        has_function_calls = False

        response = openai_client.responses.create(
            model=model,
            input=messages,
            tools=[search_tool]
        )

        messages.extend(response.output)

        for item in response.output:
            if item.type == "function_call":
                print("function_call:", item.name, item.arguments)
                call_output = make_call(item)
                messages.append(call_output)
                has_function_calls = True

            elif item.type == "message":
                print("ASSISTANT:")
                last_answer = item.content[0].text
                print(item.content[0].text)

        it = it + 1
        if has_function_calls == False:
            break

    return last_answer

In [30]:
agent_loop(instructions, "I just discovered the course. Can I still join it?")

iteration #1...
function_call: search {"query":"join course late discovered can I still join registration enrollment late join"}
iteration #2...
ASSISTANT:
Yes — you can still join the course.  

If you want to receive a certificate, make sure you submit your project while submissions are still being accepted.

Is there anything else about the course you’d like to explore?


'Yes — you can still join the course.  \n\nIf you want to receive a certificate, make sure you submit your project while submissions are still being accepted.\n\nIs there anything else about the course you’d like to explore?'

In [31]:
agent_loop(instructions, "what's queen gambit?")

iteration #1...
function_call: search {"query":"queen gambit"}
iteration #2...
function_call: search {"query":"queens gambit course FAQ"}
iteration #3...
ASSISTANT:
I couldn’t find anything in the course FAQ about “queen gambit,” so it’s likely off-topic for this course.

If you meant something else course-related, feel free to rephrase and I can check the FAQ again. Are there other areas you want to explore?


'I couldn’t find anything in the course FAQ about “queen gambit,” so it’s likely off-topic for this course.\n\nIf you meant something else course-related, feel free to rephrase and I can check the FAQ again. Are there other areas you want to explore?'

In [32]:
!uv add toyaikit

Resolved 127 packages in 4.14s                                       
⠙ Preparing packages... (0/7)                                                   ⠋ Preparing packages... (0/0)                                                   
⠙ Preparing packages... (0/7)-------------------     0 B/18.22 KiB           
⠙ Preparing packages... (0/7)-------------------     0 B/18.22 KiB           
truststore           ------------------------------     0 B/18.22 KiB
⠙ Preparing packages... (0/7)-------------------     0 B/21.96 KiB           
truststore           ------------------------------     0 B/18.22 KiB
⠙ Preparing packages... (0/7)---------- 14.79 KiB/21.96 KiB         
truststore           ------------------------------ 16.00 KiB/18.22 KiB
⠙ Preparing packages... (0/7)---------- 14.79 KiB/21.96 KiB         
truststore           ------------------------------ 16.00 KiB/18.22 KiB
⠙ Preparing packages... (0/7)---------- 14.79 KiB/21.96 KiB         
truststore           -----------------------

In [33]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [34]:
agent_tools = Tools()
agent_tools.add_tool(search, search_tool)

In [36]:
def search(query: str) -> dict[str, str]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
        boost_dict={"question": 3.0, "section": 0.5},
        filter_dict={"course": "llm-zoomcamp"}
    )

In [37]:
agent_tools = Tools()
agent_tools.add_tool(search)

In [38]:
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the FAQ database for entries matching the given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [42]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

In [43]:
runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model="gpt-5.4-mini"),
)

In [44]:
result = runner.loop(
    prompt="How do I run Olama locally?",
    callback=callback,
)

-> Response received


-> Response received


-> Response received


In [45]:
result.cost

CostInfo(input_cost=Decimal('0.0026835'), output_cost=Decimal('0.0013365'), total_cost=Decimal('0.0040200'))

In [46]:
result.all_messages

[EasyInputMessage(content="You're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches. First perform search, analyze the results \nand then perform more searches.\n\nThe question has to be about the course or its logistics, offtopic questions \nshouldn't be answered. If the search returns nothing, it's likely an off-topic question.\nIf you can't answer the question using FAQ, don't do it yourself. Only use the \nfacts from the FAQ database.\n\nAt the end, ask if there are other areas that the user wants to explore.", role='developer', phase=None, type=None),
 EasyInputMessage(content='How do I run Olama locally?', role='user', phase=None, type=None),
 ResponseFunctionToolCall(arguments='{"query":"Olama locally run local setup install ollama"}', call_id='call_

In [49]:
result2 = runner.loop(
    prompt="How do I run a different model?",
    previous_messages=result.all_messages,
    callback=callback,
)

-> Response received


-> Response received


In [50]:
runner.run()

-> Response received


-> Response received


-> Response received


Chat ended.


LoopResult(new_messages=[EasyInputMessage(content="You're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches. First perform search, analyze the results \nand then perform more searches.\n\nThe question has to be about the course or its logistics, offtopic questions \nshouldn't be answered. If the search returns nothing, it's likely an off-topic question.\nIf you can't answer the question using FAQ, don't do it yourself. Only use the \nfacts from the FAQ database.\n\nAt the end, ask if there are other areas that the user wants to explore.", role='developer', phase=None, type=None), EasyInputMessage(content='how do I run ollama in docker?', role='user', phase=None, type=None), ResponseFunctionToolCall(arguments='{"query":"run ollama in docker"}', call_id='call